In [1]:
from keras import models
from keras import layers
import numpy as np
from keras.datasets import boston_housing

In [2]:
# Load Boston housing dataset
(train_data, train_labels), (test_data, test_labels) = boston_housing.load_data()

In [3]:
# Data Preprocessing to standardize the features
# Calculate the mean and std deviation for each feature in the training data
mean = train_data.mean(axis=0)
train_data -= mean  # Subtract mean from training data to center the data
std = train_data.std(axis=0)
train_data /= std   # Divide by std deviation to standardize training data

In [4]:
# Apply the same mean and std to the test data (don't recalculate for test data)
test_data -= mean
test_data /= std

In [5]:
train_data.shape

(404, 13)

In [6]:
len(train_labels)

404

In [7]:
# Model Building function
def build_model():
    # Build a Sequential neural network model
    model = models.Sequential()
    # First hidden layer: 64 neurons, relu activation, input shape is the number of features
    model.add(layers.Dense(64, activation='relu', input_shape=(train_data.shape[1],)))
    # Second hidden layer: 64 neurons, relu activation
    model.add(layers.Dense(64, activation='relu'))
    # Output layer: 1 neuron for regression output (no activation)
    model.add(layers.Dense(1))
    # Compile model: Use mean squared error loss and rmsprop optimizer
    model.compile(optimizer='rmsprop', loss='mse', metrics=['mse'])
    return model

In [8]:
# K-fold cross-validation
k = 4  # Number of folds for cross-validation
num_of_samples = len(train_data) // k  # Number of samples per fold
epochs = 100  # Number of training epochs
all_scores = []  # List to store the validation MSE for each fold

In [9]:

# Perform K-fold cross-validation
for index in range(k):
    # Validation data for the current fold
    val_data = train_data[index * num_of_samples:(index + 1) * num_of_samples]
    val_targets = train_labels[index * num_of_samples:(index + 1) * num_of_samples]
    
    # Training data for the current fold (exclude validation fold)
    partial_train_data = np.concatenate(
        [train_data[:index * num_of_samples], train_data[(index + 1) * num_of_samples:]],
        axis=0
    )
    partial_train_labels = np.concatenate(
        [train_labels[:index * num_of_samples], train_labels[(index + 1) * num_of_samples:]],
        axis=0
    )

    # Build and train the model on the training data
    model = build_model()
    model.fit(partial_train_data, partial_train_labels, epochs=epochs, batch_size=16, verbose=0)
    
    # Evaluate the model on validation data for this fold
    val_mse, _ = model.evaluate(val_data, val_targets, verbose=0)
    
    # Store the validation MSE for this fold
    all_scores.append(val_mse)

c:\Users\PMLS\Desktop\Projects\data science\dl\env\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# Output the validation MSE scores for all folds and the average
print(f'Validation MSE scores: {all_scores}')
print(f'Average validation MSE: {np.mean(all_scores)}')

Validation MSE scores: [7.264804840087891, 11.382031440734863, 13.615509986877441, 10.691495895385742]
Average validation MSE: 10.738460540771484


In [11]:
# Optional: Train the model on the full training data for final evaluation
model = build_model()
model.fit(train_data, train_labels, epochs=epochs, batch_size=16, verbose=0)

In [12]:
# Evaluate the model on the test set
test_mse, _ = model.evaluate(test_data, test_labels)
print(f'Test MSE: {test_mse}')


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 11.6885 - mse: 11.6885
Test MSE: 14.493029594421387
